<a href="https://colab.research.google.com/github/Meyem4/lesy_test/blob/main/Colab_v%C3%A1s_v%C3%ADt%C3%A1!.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas openpyxl matplotlib seaborn

In [2]:
!pip install -U transformers accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 33.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.3/608.3 kB 20.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 64.4 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<

In [3]:
import transformers
print(transformers.__version__)

5.23.0


In [4]:
import transformers
import inspect

from transformers import TrainingArguments

print("Transformers version:", transformers.__version__)
print("Transformers location:", transformers.__file__)

params = inspect.signature(TrainingArguments.__init__).parameters

print("\nWarmup ratio supported:", "warmup_ratio" in params)
print("Eval strategy supported:", "eval_strategy" in params)
print("Evaluation strategy supported:", "evaluation_strategy" in params)

Transformers version: 5.23.0
Transformers location: /usr/local/lib/python3.13/dist-packages/transformers/__init__.py

Warmup ratio supported: False
Eval strategy supported: True
Evaluation strategy supported: False


In [7]:
import numpy as np
import pandas as pd
# Load dataset
df = pd.read_excel("/content/czech_reviews.xlsx")

In [8]:
import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)

from datasets import Dataset

# Reproducibility
SEED = 42

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Load dataset
df = pd.read_excel("/content/czech_reviews.xlsx")

# Remove missing and empty reviews BEFORE splitting
df = df.dropna(subset=["review_czech", "label"]).copy()

df["review_czech"] = df["review_czech"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip().str.upper()

df = df[df["review_czech"] != ""].reset_index(drop=True)

# Define sentiment labels
label2id = {
    "NEGATIVE": 0,
    "NEUTRAL": 1,
    "POSITIVE": 2
}

id2label = {
    0: "NEGATIVE",
    1: "NEUTRAL",
    2: "POSITIVE"
}

# Check labels
unknown_labels = set(df["label"].unique()) - set(label2id)

if unknown_labels:
    raise ValueError(f"Unknown labels found: {unknown_labels}")

df["labels"] = df["label"].map(label2id).astype(int)

print("Dataset size:", len(df))
print("\nClass distribution:")
print(df["label"].value_counts())

print("\nClass distribution (%):")
print(df["label"].value_counts(normalize=True) * 100)

Dataset size: 22808

Class distribution:
label
POSITIVE    11215
NEGATIVE     5939
NEUTRAL      5654
Name: count, dtype: int64

Class distribution (%):
label
POSITIVE    49.171343
NEGATIVE    26.039109
NEUTRAL     24.789548
Name: proportion, dtype: float64


In [9]:
# First: separate 15% test data
train_val_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=SEED,
    stratify=df["labels"]
)

# Second: separate 15% validation data
# 0.17647 of the remaining 85% = 15% of the original dataset
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.17647,
    random_state=SEED,
    stratify=train_val_df["labels"]
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTraining distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

print("\nTest distribution:")
print(test_df["label"].value_counts())

Training: 15964
Validation: 3422
Test: 3422

Training distribution:
label
POSITIVE    7849
NEGATIVE    4157
NEUTRAL     3958
Name: count, dtype: int64

Validation distribution:
label
POSITIVE    1683
NEGATIVE     891
NEUTRAL      848
Name: count, dtype: int64

Test distribution:
label
POSITIVE    1683
NEGATIVE     891
NEUTRAL      848
Name: count, dtype: int64


In [10]:
MODEL_NAME = "ufal/robeczech-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Model:", MODEL_NAME)

config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/955k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/601k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.66M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/963 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  504MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: ufal/robeczech-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda
Model: ufal/robeczech-base


In [11]:
train_dataset = Dataset.from_pandas(
    train_df[["review_czech", "labels"]].reset_index(drop=True)
)

val_dataset = Dataset.from_pandas(
    val_df[["review_czech", "labels"]].reset_index(drop=True)
)

test_dataset = Dataset.from_pandas(
    test_df[["review_czech", "labels"]].reset_index(drop=True)
)

MAX_LENGTH = 256

def tokenize_function(examples):
    return tokenizer(
        examples["review_czech"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["review_czech"]
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["review_czech"]
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["review_czech"]
)

# Dynamic padding
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Tokenization completed.")

Map:   0%|          | 0/15964 [00:00<?, ? examples/s]

Map:   0%|          | 0/3422 [00:00<?, ? examples/s]

Map:   0%|          | 0/3422 [00:00<?, ? examples/s]

Tokenization completed.


In [12]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        labels=[0, 1, 2],
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "macro_f1": np.mean(f1),
        "macro_precision": np.mean(precision),
        "macro_recall": np.mean(recall),

        "negative_f1": f1[0],
        "neutral_f1": f1[1],
        "positive_f1": f1[2]
    }

In [13]:

import inspect
import torch

from transformers import TrainingArguments

params = inspect.signature(
    TrainingArguments.__init__
).parameters

args = {
    "output_dir": "./robeczech_sentiment",
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "num_train_epochs": 6,
    "lr_scheduler_type": "cosine",
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_accuracy",
    "greater_is_better": True,
    "save_total_limit": 2,
    "logging_strategy": "epoch",
    "report_to": "none",
    "fp16": torch.cuda.is_available(),
    "seed": 42,
    "data_seed": 42,
    "dataloader_num_workers": 0
}

# Use supported warmup configuration
if "warmup_ratio" in params:
    args["warmup_ratio"] = 0.1
elif "warmup_steps" in params:
    args["warmup_steps"] = 100

# Support different Transformers API versions
if "eval_strategy" in params:
    args["eval_strategy"] = "epoch"
elif "evaluation_strategy" in params:
    args["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**args)

print("Training arguments configured successfully.")


Training arguments configured successfully.


In [14]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.001
        )
    ]
)

# Start training
train_result = trainer.train()

print("\nTraining completed.")

print("\nBest checkpoint:")
print(trainer.state.best_model_checkpoint)

print("\nBest validation accuracy:")
print(trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall,Negative F1,Neutral F1,Positive F1
1,0.639492,0.551025,0.772648,0.729998,0.745196,0.721837,0.747356,0.571429,0.871210
2,0.487168,0.578092,0.767680,0.743087,0.752706,0.743751,0.743136,0.618257,0.867867
3,0.388706,0.623143,0.783460,0.737199,0.750902,0.736110,0.765180,0.561813,0.884604
4,0.299492,0.675122,0.790473,0.759863,0.764499,0.756937,0.761792,0.631519,0.886277
5,0.233681,0.769272,0.786674,0.754347,0.760430,0.750205,0.757558,0.621449,0.884033
6,0.200876,0.791876,0.786382,0.752343,0.757618,0.748238,0.759346,0.612922,0.884761


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed.

Best checkpoint:
./robeczech_sentiment/checkpoint-3992

Best validation accuracy:
0.7904734073641145


In [15]:
print(train_df["label"].value_counts())
print(train_df["label"].value_counts(normalize=True) * 100)

label
POSITIVE    7849
NEGATIVE    4157
NEUTRAL     3958
Name: count, dtype: int64
label
POSITIVE    49.166875
NEGATIVE    26.039840
NEUTRAL     24.793285
Name: proportion, dtype: float64


In [16]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

output = trainer.predict(val_dataset)

y_true = output.label_ids
y_pred = np.argmax(output.predictions, axis=1)

print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=["NEGATIVE", "NEUTRAL", "POSITIVE"]
))

print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2]))

              precision    recall  f1-score   support

    NEGATIVE       0.80      0.73      0.76       891
     NEUTRAL       0.61      0.65      0.63       848
    POSITIVE       0.88      0.90      0.89      1683

    accuracy                           0.79      3422
   macro avg       0.76      0.76      0.76      3422
weighted avg       0.79      0.79      0.79      3422

[[ 646  191   54]
 [ 139  551  158]
 [  20  155 1508]]


In [17]:
import numpy as np
import pandas as pd

output = trainer.predict(val_dataset)

y_true = output.label_ids
y_pred = np.argmax(output.predictions, axis=1)

results = val_df.copy().reset_index(drop=True)

results["actual"] = y_true
results["predicted"] = y_pred

# Actual NEGATIVE or POSITIVE, but predicted NEUTRAL
false_neutral = results[
    (results["actual"] != 1) &
    (results["predicted"] == 1)
].copy()

print("False NEUTRAL predictions:", len(false_neutral))

display(
    false_neutral[
        ["review_czech", "actual", "predicted"]
    ].head(50)
)

False NEUTRAL predictions: 346


,review_czech,actual,predicted
7,"Prkne místo, I stánek s menším občerstvením",2,1
16,Kemp je krasny a dobre vybaveny. Kamen urazu s...,0,1
28,"Pěkný výhled, ale moc mě to místo nenadchlo.",0,1
56,Okolí nic moc,0,1
87,"Moc tu k vyprávění nemám, jediný můj zážitek z...",2,1
104,Kemp je pěkný a čistý ale pokud chcete se vyko...,2,1
140,"Není to kemping, je to jen tábor! Vodní bloky ...",0,1
149,Jen kdyby tam nebylo tolik lidí.....,2,1
152,Pivo dobry.Jidlo taky.Cene su ok jen ta obshul...,0,1
168,"Jídlo super, pivo nadprůmer, ale nebrat v roce...",0,1


In [19]:
from google.colab import files

# Export false NEUTRAL predictions to Excel
false_neutral.to_excel(
    "/content/false_neutral_reviews.xlsx",
    index=False
)

# Download the Excel file
files.download("/content/false_neutral_reviews.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
import numpy as np
import pandas as pd

# Get predictions for the validation dataset
val_output = trainer.predict(val_dataset)

# Convert model logits to predicted class IDs
val_pred = np.argmax(val_output.predictions, axis=1)
val_true = val_output.label_ids

# Create a readable results DataFrame
val_results_df = val_df.copy().reset_index(drop=True)

id2label = {
    0: "NEGATIVE",
    1: "NEUTRAL",
    2: "POSITIVE"
}

val_results_df["actual"] = [id2label[int(x)] for x in val_true]
val_results_df["predicted"] = [id2label[int(x)] for x in val_pred]

print("Validation predictions created.")

Validation predictions created.


In [17]:
neutral_errors = val_results_df[
    (val_results_df["actual"] == "NEUTRAL") &
    (val_results_df["predicted"] != "NEUTRAL")
].copy()

print("Number of misclassified neutral reviews:", len(neutral_errors))

display(
    neutral_errors[["review_czech", "actual", "predicted"]]
    .head(50)
)

Number of misclassified neutral reviews: 350


,review_czech,actual,predicted
5,Dobře.,NEUTRAL,NEGATIVE
19,"Krasne misto, ale mnoho bezohledných lidi.",NEUTRAL,POSITIVE
22,"Kdyby bylo vidět, určitě by se mi tam libilo ...",NEUTRAL,POSITIVE
31,"úžasné místo, jen už je zde vidět zub času.......",NEUTRAL,POSITIVE
45,"Pěkná rozhledna ze které,ale není nic vidět",NEUTRAL,POSITIVE
63,到České Budějovice 遊覧，必到景點是Náměstí Přemysla Ota...,NEUTRAL,POSITIVE
67,Oproti skanzenu v Rožnově pod Radhoštěm nesrov...,NEUTRAL,NEGATIVE
73,Nepřehledná.,NEUTRAL,NEGATIVE
119,Nicht sonderlich aufregend aber für einen Tag...,NEUTRAL,POSITIVE
135,Po červené na kole masakr ale jinak pěkná rozh...,NEUTRAL,POSITIVE


In [18]:
neutral_as_negative = neutral_errors[
    neutral_errors["predicted"] == "NEGATIVE"
]

neutral_as_positive = neutral_errors[
    neutral_errors["predicted"] == "POSITIVE"
]

print("Neutral predicted as negative:", len(neutral_as_negative))
print("Neutral predicted as positive:", len(neutral_as_positive))

Neutral predicted as negative: 124
Neutral predicted as positive: 226


In [20]:
neutral_errors.to_excel(
    "/content/neutral_misclassified_reviews.xlsx",
    index=False
)

print("File saved: /content/neutral_misclassified_reviews.xlsx")

File saved: /content/neutral_misclassified_reviews.xlsx
